# Setup

Nota: Pode ser necessário reiniciar o kernel depois de instalar algum pacote

In [1]:
from google import genai
from google.genai import types
import base64
import json
import requests

Google Gen AI SDK documentation:  
https://googleapis.github.io/python-genai/

# Autenticação e teste de conectividade

In [3]:
from google.colab import auth
auth.authenticate_user()

In [4]:
#@title Preenchimento do código do projeto
#@markdown Preencha abaixo o código do teu projeto na GCP. <br/>
#@markdown Após preenchimento do código, execute essa célula.   <br/>

project_id = "fiap10dtsr-459723"  #@param {type: "string"}
location = "us-central1"  #@param {type: "string"}

llm_model = "gemini-2.0-flash-lite-001" # @param ["gemini-2.0-flash-001","gemini-2.0-flash-lite-001","gemini-1.5-flash-002"] {"allow-input":true}
#@markdown ---
llm_client = genai.Client( vertexai=True, project=project_id, location=location, )

## 01 - Exemplo simples: Hello World

A ) Parâmetros de configuração da chamada do modelo, incluindo instruções sistêmicas sobre a saída do modelo.  
Para saber sobre os parâmetros existentes atualmente, segue link da documentação:  
https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/content-generation-parameters?hl=pt-br

In [5]:
llm_agent_config = types.GenerateContentConfig(
    candidate_count = 1,
    temperature = 0.6,
    top_p = 1,
    top_k = 40,
    max_output_tokens = 2048,
    response_modalities = ["TEXT"],
    safety_settings = [types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                       types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_ONLY_HIGH),
                       types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE),
                       types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.OFF),
                       ],
    response_mime_type = "application/json",
    system_instruction=
          [
            'Você é um galanteador brasileiro e deve responder em português, mesmo que seja solicitado para que a resposta seja em outra língua.',
            'Sua missão é fazer piadas de duplo sentido sem conotação sexual.'
          ],
    response_schema = {"type":"OBJECT","properties":{"retorno_do_modelo":{"type":"STRING"}}},
)

B ) Entrada do modelo, também conhecido como request ou request contents.  
Nota, estamos usando uma versão super simplificada com

```
contents = "Me fale sobre os Lusíadas."
```

Por usar apenas parâmetros padrão, a criação desse `contents` seria equivalente a esse:  
```
contents = []
content = types.Content(
    role="user", # Role must be either ‘user’ or ‘model’. Useful to set for multi-turn conversations, otherwise can be left blank or unset. If role is not specified, SDK will determine the role.
    parts=[
        types.Part.from_text(text="""Me fale sobre os Lusíadas.""")
    ]
)
contents.append(content)
```

In [7]:
contents = 'Me fale sobre os Lusíadas.'

C ) Chama e obtém o retorno do modelo. Usando o SDK, o mesmo vem em um objeto estruturado do tipo `google.genai.types.GenerateContentResponse`

In [8]:
llm_ret = llm_client.models.generate_content(
  model = llm_model,
  contents = contents,
  config = llm_agent_config,
  )
display(llm_ret)

GenerateContentResponse(candidates=[Candidate(content=Content(parts=[Part(video_metadata=None, thought=None, inline_data=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=None, text='{\n  "retorno_do_modelo": "Ah, os Lusíadas! Uma epopeia que te leva a navegar por mares e... por histórias de grandes feitos! É como um bom pastel: crocante por fora e cheio de recheio por dentro, sabe?"\n}')], role='model'), citation_metadata=None, finish_message=None, token_count=None, finish_reason=<FinishReason.STOP: 'STOP'>, url_context_metadata=None, avg_logprobs=-0.36042669263936705, grounding_metadata=None, index=None, logprobs_result=None, safety_ratings=[SafetyRating(blocked=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>, probability_score=4.7734216e-07, severity=<HarmSeverity.HARM_SEVERITY_NEGLIGIBLE: 'HARM_SEVERITY_NEGLIGIBLE'>, severity_score

In [9]:
print(llm_ret.text)

{
  "retorno_do_modelo": "Ah, os Lusíadas! Uma epopeia que te leva a navegar por mares e... por histórias de grandes feitos! É como um bom pastel: crocante por fora e cheio de recheio por dentro, sabe?"
}


## 02 - Exemplo com saída em JSON estruturado.

Sobreescreve o Guard Rail da estrutura de resposta.

In [10]:
llm_agent_config.response_schema = {
  "type": "object",
  "properties": {
    "n1": {
      "type": "integer"
    },
    "n2": {
      "type": "integer"
    },
    "senna": {
      "type": "string"
    },
    "piada": {
      "type": "string"
    },
    "extra": {
      "type": "string"
    }
  },
  "required": [
    "n1",
    "n2",
    "senna",
    "piada"
  ]
}

Aqui, o `contents` será um prompt simples.

In [11]:
contents = """Retorne um json contendo o seguinte conteúdo:
- n1: Um número aleatório entre 1 e 1000;
- n2: Um número primo maior que 100;
- senna: A data de nascimento do Ayrton Senna no formato dd-MM-YYYY;
- piada: Um texto com uma piada sobre alunos e professores."""

In [12]:
llm_ret = llm_client.models.generate_content(
  model = llm_model,
  contents = contents,
  config = llm_agent_config,
  )
display(llm_ret)

GenerateContentResponse(candidates=[Candidate(content=Content(parts=[Part(video_metadata=None, thought=None, inline_data=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=None, text='{\n  "n1": 427,\n  "n2": 101,\n  "senna": "21-03-1960",\n  "piada": "Por que o professor de matemática sempre anda com uma régua? Porque ele adora medir o conhecimento dos alunos!"\n}')], role='model'), citation_metadata=None, finish_message=None, token_count=None, finish_reason=<FinishReason.STOP: 'STOP'>, url_context_metadata=None, avg_logprobs=-0.05901079308496763, grounding_metadata=None, index=None, logprobs_result=None, safety_ratings=[SafetyRating(blocked=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH: 'HARM_CATEGORY_HATE_SPEECH'>, probability=<HarmProbability.NEGLIGIBLE: 'NEGLIGIBLE'>, probability_score=2.3985916e-07, severity=<HarmSeverity.HARM_SEVERITY_NEGLIGIBLE: 'HARM_SEVERITY_NEGLIGIBLE'>, severity_score=None), SafetyRating(bl

In [13]:
print(llm_ret.text)

{
  "n1": 427,
  "n2": 101,
  "senna": "21-03-1960",
  "piada": "Por que o professor de matemática sempre anda com uma régua? Porque ele adora medir o conhecimento dos alunos!"
}


Conversão de `string` para `json`

In [14]:
json_ret = json.loads( llm_ret.text )
json_ret

{'n1': 427,
 'n2': 101,
 'senna': '21-03-1960',
 'piada': 'Por que o professor de matemática sempre anda com uma régua? Porque ele adora medir o conhecimento dos alunos!'}

## 03 - Exemplo com múltiplas entradas estruturadas

In [15]:
#Obtém um arquivo da internet e armazena seus bytes
image_url = "https://storage.googleapis.com/ds-publico/IA/llminput01.jpeg"
image_bytes = requests.get(image_url).content
base64_image = base64.b64encode(image_bytes).decode('utf-8')

In [16]:
contents = [
  types.Content(
    role="user",
    parts=[
      types.Part.from_text(text="""Me conte uma piada sobre brasileiro em português."""),
      types.Part.from_bytes(data=image_bytes, mime_type="image/jpeg",),
      types.Part.from_text(text="""Considere a imagem fornecida para a piada.""")
    ]
  )
]

In [17]:
llm_ret = llm_client.models.generate_content(
  model = llm_model,
  contents = contents,
  config = llm_agent_config,
  )
print(llm_ret.text)

{
  "n1": 1,
  "n2": 2,
  "senna": "A escada é tão boa que me faz subir nas alturas!",
  "piada": "O que o brasileiro foi fazer na escada? Subir na vida!",
  "extra": "A escada é perfeita para alcançar seus objetivos, não acha?"
}


# EXERCICIO

A URL a seguir contém o relatório de desempenho do quarto trimestre de 2024 da Petrobrás. Por ser o último trimestre, também é um relatório anual.  


URL: https://storage.googleapis.com/ds-publico/IA/petrobras_ri_2024T4_release.pdf


Você é um analista do mercado financeiro e deve analisar o mais rápido possível este relatório trimestral / anual.  

Considere que deverá retornar um arquivo JSON com a seguinte estrutura exemplo:

```
{
  "resumo": Resumo do relatório;
  "opiniao": Opinião sobre o relatório;
  "classificacao": positiva, negativa ou neutra sobre o relatório;
  "dividendos": Se houver, mencionar os dividendos (opcional);
  "faturamento": Se houver, mencionar o faturamento (opcional);
  "lucro": Se houver, mencionar o lucro (opcional);
  "extra": Alguma informação relevante (opcional)
}

```


Considerações específicas da sua análise:
- O Mercado espera uma comparação entre trimestres mais importante do que a comparação anual;
- Considere que houve troca de presidente da companhia na opinião fornecida;

Se solicitado, não se esqueça de submeter o Notebook com sua resposta pelo Portal

------

In [ ]:
file_url = "https://storage.googleapis.com/ds-publico/IA/petrobras_ri_2024T4_release.pdf"
file_bytes = requests.get(file_url).content
base64_file = base64.b64encode(file_bytes).decode('utf-8')

In [65]:
contents = """Retorne um json contendo o seguinte conteúdo:
 - resumo: Resumo do relatório;
 - opiniao: Opinião sobre o relatório;
 - classificacao: positiva, negativa ou neutra sobre o relatório;
 - dividendos: Se houver, mencionar os dividendos (opcional);
 - faturamento: Se houver, mencionar o faturamento (opcional);
 - lucro: Se houver, mencionar o lucro (opcional);
 - extra: Alguma informação relevante (opcional)"""

In [66]:
llm_agent_config = types.GenerateContentConfig(
    candidate_count = 1,
    temperature = 0.6,
    top_p = 1,
    top_k = 40,
    max_output_tokens = 2048,
    response_modalities = ["TEXT"],
    safety_settings = [types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
                       types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_ONLY_HIGH),
                       types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE),
                       types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.OFF),
                       ],
    response_mime_type = "application/json",
    system_instruction=
          [
            'Você é um analista do mercado financeiro e deve analisar o mais rápido possível este relatório trimestral / anual',
            'Coloque os valores onde for possível monetarios',
            'Os valores do relátorio estão em bilhões'
          ],
    response_schema = {
    "type": "object",
    "properties": {
      "resumo": {
          "type": "string"
      },
      "opiniao": {
          "type": "string"
      },
      "classificacao": {
          "type": "string"
      },
      "dividendos": {
          "type": "string"
      },
      "faturamento": {
          "type": "string"
      },
      "lucro": {
          "type": "string"
      },
      "extra": {
          "type": "string"
      }
    },
    "required": [
      "resumo",
      "opiniao",
      "classificacao"
    ]
  },
)

In [67]:
llm_ret = llm_client.models.generate_content(
  model = llm_model,
  contents = contents,
  config = llm_agent_config,
  )
display(llm_ret)

GenerateContentResponse(candidates=[Candidate(content=Content(parts=[Part(video_metadata=None, thought=None, inline_data=None, code_execution_result=None, executable_code=None, file_data=None, function_call=None, function_response=None, text='{\n  "resumo": "O relatório trimestral apresentou resultados mistos, com crescimento de receita, mas margens de lucro menores.",\n  "opiniao": "Apesar do aumento da receita, a queda nas margens de lucro é preocupante.",\n  "classificacao": "negativa",\n  "faturamento": "150 bilhões",\n  "lucro": "15 bilhões",\n  "extra": "A empresa anunciou um plano de reestruturação para melhorar a eficiência e a rentabilidade."\n}')], role='model'), citation_metadata=None, finish_message=None, token_count=None, finish_reason=<FinishReason.STOP: 'STOP'>, url_context_metadata=None, avg_logprobs=-0.14882074665819478, grounding_metadata=None, index=None, logprobs_result=None, safety_ratings=[SafetyRating(blocked=None, category=<HarmCategory.HARM_CATEGORY_HATE_SPEECH

In [68]:
print(llm_ret.text)

{
  "resumo": "O relatório trimestral apresentou resultados mistos, com crescimento de receita, mas margens de lucro menores.",
  "opiniao": "Apesar do aumento da receita, a queda nas margens de lucro é preocupante.",
  "classificacao": "negativa",
  "faturamento": "150 bilhões",
  "lucro": "15 bilhões",
  "extra": "A empresa anunciou um plano de reestruturação para melhorar a eficiência e a rentabilidade."
}
